In [ ]:
import ollama
import sys
import redis
import json
import asyncio
from mcp import ClientSession
from mcp.client.sse import sse_client

In [ ]:
LARGE_MODEL='llama3.2:3b'

In [ ]:
LOCAL_SERVER_URL = "http://127.0.0.1:54321/sse"

In [ ]:
START_FRESH = False  # Set to True to clear memory at the start of the session

Connect to Redis (Ensure your Redis server is running!)

In [ ]:
# We use decode_responses=True to handle strings easily
r = redis.Redis(host='localhost', port=6379, db=0, decode_responses=True)

The code block below defines two utility functions for managing chat history sessions stored in Redis.

1.  `list_active_sessions()`
- **Purpose**: Lists all active chat history sessions by querying Redis for keys matching the pattern `"chat_history:*"`.
- **Behavior**: Retrieves the keys using `r.keys()`, then prints them as a comma-separated list if any exist; otherwise, prints "Cache is empty."
- **Usage**: Call this function to inspect what memory keys are currently stored in Redis.

2. `clear_session(session_id="user_123")`
- **Purpose**: Clears the chat history for a specific session by deleting its corresponding Redis key.
- **Behavior**: Constructs the memory key as `f"chat_history:{session_id}"`, deletes it using `r.delete()`, and prints a confirmation message with an emoji.
- **Usage**: Call this function with a session ID (default is "user_123") to wipe that session's memory and start fresh.

In [ ]:
def list_active_sessions():
    """Run this to see what memory keys exist in Redis."""
    keys = r.keys("chat_history:*")
    print(f"Active Sessions: {', '.join(keys)}" if keys else "Cache is empty.")

def clear_session(session_id="user_123"):
    """Run this to wipe a specific project and start over."""
    memory_key = f"chat_history:{session_id}"
    r.delete(memory_key)
    print(f"🧹 Wiped memory for: {session_id}")

#### Explanation of `summarize_and_save_memory` Function

This asynchronous function manages conversation history by summarizing it when it becomes too long and then saving the updated history to Redis for persistence.

Key Steps:
1. **Check History Length**: If the number of messages exceeds 10, it triggers summarization to keep the history concise.
2. **Summarization Process**:
    - Extracts the first 6 messages (older context) for summarization.
    - Retains the last 4 messages (recent context) unchanged.
    - Uses the LLM to generate a summary of the older messages via a prompt.
    - Creates a new "system" message containing the summary to replace the older context.
    - Combines the summary message with the recent messages to form the new history.
3. **Save to Redis**: Converts messages to a serializable format (using `.model_dump()` if available) and stores the JSON-serialized history in Redis under the provided `memory_key`.

#### Purpose:
- Prevents memory bloat by compressing early conversation parts.
- Ensures efficient storage and retrieval of chat history across sessions.
- Logs progress to stderr for debugging without interrupting user output.

In [ ]:
async def summarize_and_save_memory(messages, model, memory_key):
    # Only summarize if history is getting too long (e.g., > 10 messages)
    if len(messages) > 10:
        # print("🧠 History too long. Summarizing early messages...")
        sys.stderr.write("🧠 History too long...\n")
        
        # We take the first 6 messages (the 'old' context)
        to_summarize = messages[:6]
        # Keep the recent 4 messages exactly as they are
        recent_messages = messages[6:]
        
        summary_prompt = {
            'role': 'user', 
            'content': f"Summarize the following conversation key points concisely: {json.dumps(to_summarize)}"
        }
        
        summary_resp = ollama.chat(model=model, messages=[summary_prompt])
        
        # Create a new 'Summary' message to act as the new foundation
        summary_message = {
            'role': 'system', 
            'content': f"Previously in this conversation: {summary_resp['message']['content']}"
        }
        
        # New history = [The Summary] + [The Recent Messages]
        messages = [summary_message] + recent_messages
        print(f"📉 Memory compressed. New length: {len(messages)}")

    # Save to Redis
    # Ensure everything is a dict using .model_dump() to avoid the TypeError
    serializable = [m.model_dump() if hasattr(m, 'model_dump') else m for m in messages]
    r.set(memory_key, json.dumps(serializable))
    return messages

#### Explanation of `background_memory_tasks` Function

This asynchronous function manages saving conversation history to Redis and summarizing it in the background to prevent blocking the main execution thread.

Key Steps:
1. **Prepare Serializable List**: Converts each message to a dictionary using `.model_dump()` if the method exists, ensuring compatibility with JSON serialization.
2. **Save to Redis**: Stores the serialized messages as a JSON string in Redis under the specified `memory_key`.
3. **Summarize if Needed**: Asynchronously calls `summarize_and_save_memory` to compress the history if it's too long (noted as the slowest part).
4. **Logging**: Outputs success or error messages to stderr to avoid interfering with user output.

Purpose:
- Persists memory efficiently without freezing the application.
- Handles exceptions by logging errors for debugging.

In [ ]:
async def background_memory_tasks(messages, model, memory_key):
    """Handles Redis saving and summarization in the background."""
    try:
        # 1. Prepare serializable list
        serializable = [
            m.model_dump() if hasattr(m, 'model_dump') else m 
            for m in messages
        ]
        
        # 2. Save to Redis
        r.set(memory_key, json.dumps(serializable))
        
        # 3. Summarize if needed (This is the slowest part!)
        await summarize_and_save_memory(messages, model, memory_key)
        
        # Log to stderr so it doesn't interrupt the user's view
        import sys
        sys.stderr.write("✅ Background: Memory synced and summarized.\n")
    except Exception as e:
        import sys
        sys.stderr.write(f"❌ Background Task Error: {e}\n")

In [ ]:
system_prompt = (
    "You are a smart assistant with perfect memory of the conversations. "
    "Rule 1: If I ask you about something we already discussed, "
    "read the conversation history and answer directly. DO NOT use tools for this. "
    "Rule 2: CRITICAL RULE: You are strictly FORBIDDEN from using the 'search_web' tool "
    "if the user's prompt asks about 'I', 'you', 'we', 'history', 'just asked', or 'said'. "
    "If the answer is in your memory, reply immediately without tools. "
    "Only use 'search_web' for real-world facts outside of this conversation."
    "Rule 3: If the conversation history gets too long, summarize the early parts to keep it concise."
    "Rule 4: Use the 'read_local_file' tool if I ask you to read a file."
    "Rule 5: ALWAYS save important facts to memory after you find them, so you can recall them later without searching again."
)

In [ ]:
async def run_mcp_query_with_memory(user_prompt, system_prompt, model, session_id=1):
    async with sse_client(LOCAL_SERVER_URL) as (read, write): 
        async with ClientSession(read, write) as session:
            await session.initialize()
            memory_key = f"chat_history:{session_id}"

            # --- 1. SILENT MEMORY MANAGEMENT ---
            # We handle Redis directly before the LLM even sees it
            if START_FRESH:
                print(f"🧹 FRESH_START is True. Wiping Redis key: {memory_key}")
                r.delete(memory_key)
                messages = [{'role': 'system', 'content': system_prompt}] 
            else:
                existing_history = r.get(memory_key)
                if existing_history:
                    messages = json.loads(existing_history)
                    print(f"🧠 Memory Loaded: {len(messages)} messages found.")
                else:
                    messages = [{'role': 'system', 'content': system_prompt}]
            
            # Add the new user prompt to history
            messages.append({'role': 'user', 'content': user_prompt})

            # --- 2. PREPARE SERVER TOOLS ---
            # Because we cleaned the server, this ONLY fetches search_web and read_local_file
            tools = await session.list_tools()
            ollama_tools = [{
                "type": "function",
                "function": {
                    "name": t.name,
                    "description": t.description,
                    "parameters": t.inputSchema,
                }
            } for t in tools.tools]

            # --- 3. FIRST LLM CALL ---
            response = ollama.chat(
                model=model, 
                messages=messages, 
                tools=ollama_tools, 
                options={
                    "num_ctx": 8192,  
                    "temperature": 0  
                }
            )
            
            # Convert the Message object to a plain dictionary and save to history
            assistant_msg = response['message'].model_dump() 
            messages.append(assistant_msg)   
             
            # --- 4. HANDLE TOOL CALLS (AGENTIC LOOP) ---
            if assistant_msg.get('tool_calls'):
                for tool_call in assistant_msg['tool_calls']:
                    t_name = tool_call['function']['name']
                    t_args = tool_call['function']['arguments']
                    
                    print(f"🛠️ Tool Call: {t_name} | Searching for: {t_args}")
                    result = await session.call_tool(t_name, t_args)
                    
                    # Add tool result to history so the LLM knows it succeeded
                    messages.append({
                        'role': 'tool', 
                        'content': str(result.content), 
                        'name': t_name
                    })
                
                # Get final thought from LLM after seeing tool results
                print("🤖 Assistant: ", end="", flush=True)
                final_content = ""

                # STREAMING THE RESPONSE
                stream = ollama.chat(model=model, messages=messages, stream=True)
                for chunk in stream:
                    content = chunk['message']['content']
                    print(content, end="", flush=True)
                    final_content += content

                print() # Print new line when stream finishes
                
                # CRITICAL FIX: Save the final generated answer to the history list!
                messages.append({'role': 'assistant', 'content': final_content})
                
            else:
                # If no tools were called, the final content is just the first response
                final_content = assistant_msg['content']
                print(f"🤖 Assistant: {final_content}")

            # --- 5. BACKGROUND MEMORY SYNC ---
            # Save the updated history (including tool results and final answer) without freezing the app
            asyncio.create_task(background_memory_tasks(messages, model, memory_key))
            
            return final_content

In [ ]:
query = "What is the capital of Germany?"
await run_mcp_query_with_memory(query, system_prompt, LARGE_MODEL)

In [ ]:
query = "What did I just ask?"
await run_mcp_query_with_memory(query, system_prompt, LARGE_MODEL)

In [ ]:
query = "who is the actor in the movie Inception ?"
print(await run_mcp_query_with_memory(query, system_prompt, LARGE_MODEL))

In [ ]:
query = "Read and summarize the contents in file ai_response.txt in the root folder"
print(await run_mcp_query_with_memory(query, system_prompt, LARGE_MODEL))

In [ ]:
query = "actor in the movie Inception?"
print(await run_mcp_query_with_memory(query, system_prompt, LARGE_MODEL))

#### wipe the memory for a fresh start

In [ ]:
# Wipe the corrupted memory
r.delete("chat_history:1")  # Make sure the session ID matches what you are using
print("Memory wiped! Ready for a fresh start.")